In [4]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd
import lightgbm as lgb
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2025
HERE = Path.cwd()
REPO = HERE.parent if HERE.name == "q5_kaggle" else HERE
DATA_DIR = REPO / "Assignment1" / "cs-610-assignment-1-question-5-2026"

train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
y_train = train["target"].values
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()

# best params from earlier Optuna run (v3 features)
best_params = {
    "learning_rate":    0.023244641257268547,
    "num_leaves":       90,
    "min_data_in_leaf": 14,
    "feature_fraction": 0.5042299812794246,
    "bagging_fraction": 0.9708522995554573,
    "bagging_freq":     4,
    "lambda_l1":        1.4345478190845018e-07,
    "lambda_l2":        3.437058206065732,
}

def featurize(df):
    out = pd.DataFrame(index=df.index)
    counts = ["favourites_count","followers_count","friends_count",
              "statuses_count","average_tweets_per_day","account_age_days"]
    for c in counts: out[c] = df[c].astype(float)
    for c in ["favourites_count","followers_count","friends_count","statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])
    for c in ["default_profile","default_profile_image","geo_enabled","verified"]:
        out[c] = df[c].astype(int)
    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"]    = (df["location"].notna() & (df["location"]!="unknown")).astype(int)
    out["has_url_bg"]      = df["profile_background_image_url"].notna().astype(int)
    out["desc_len"]  = df["description"].fillna("").str.len()
    out["sn_len"]    = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1))
    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")
    out["followers_per_friend"] = df["followers_count"]/df["friends_count"].clip(lower=1)
    out["statuses_per_day"]     = df["statuses_count"] /df["account_age_days"].clip(lower=1)
    ct = pd.to_datetime(df["created_at"])
    out["created_year"]  = ct.dt.year.astype(float)
    out["created_month"] = ct.dt.month.astype(float)
    out["created_dow"]   = ct.dt.dayofweek.astype(float)
    out["created_hour"]  = ct.dt.hour.astype(float)
    out["hour_sin"] = np.sin(2*np.pi*out["created_hour"]/24)
    out["hour_cos"] = np.cos(2*np.pi*out["created_hour"]/24)
    out["dow_sin"]  = np.sin(2*np.pi*out["created_dow"]/7)
    out["dow_cos"]  = np.cos(2*np.pi*out["created_dow"]/7)
    return out

X_v3_train = featurize(train); X_v3_test = featurize(test)
numeric_cols_v3 = [c for c in X_v3_train.columns if c != "lang"]

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=5, max_df=0.95,
                        lowercase=True, sublinear_tf=True, strip_accents="unicode")
Xt_train = tfidf.fit_transform(train["description"].fillna(""))
Xt_test  = tfidf.transform(test["description"].fillna(""))

oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_v3_train[["lang"]])
lang_test_oh  = oh.transform(X_v3_test[["lang"]])

num_train_v3 = sparse.csr_matrix(X_v3_train[numeric_cols_v3].fillna(0).values)
num_test_v3  = sparse.csr_matrix(X_v3_test[numeric_cols_v3].fillna(0).values)

X_combined_train_v3 = sparse.hstack([num_train_v3, lang_train_oh, Xt_train]).tocsr()
X_combined_test_v3  = sparse.hstack([num_test_v3,  lang_test_oh,  Xt_test ]).tocsr()

def lgbm_cv(X, y, n_splits=5, params=None, verbose=True):
    p = dict(n_estimators=2000, n_jobs=-1, verbose=-1)
    if params: p.update(params)
    p["random_state"] = p.get("random_state", RANDOM_STATE)
    take = lambda M, idx: M.iloc[idx] if hasattr(M, "iloc") else M[idx]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    oof = np.zeros(X.shape[0])
    for fold, (tr, va) in enumerate(skf.split(np.zeros(X.shape[0]), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr], eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        oof[va] = clf.predict_proba(take(X, va))[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        iters.append(clf.best_iteration_)
        if verbose: print(f"Fold {fold+1}: AUC={aucs[-1]:.4f}  iter={iters[-1]}")
    if verbose:
        print(f"\nMean fold AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"OOF AUC      : {roc_auc_score(y, oof):.4f}")
    return {"oof": oof, "aucs": aucs, "iters": iters}

# Compute v4 baseline OOF for comparison (this gives us 0.9452 reference)
print("Computing v4 baseline OOF for reference...")
cv_v4 = lgbm_cv(X_combined_train_v3, y_train, params=best_params, verbose=False)
lgbm_v4_oof = cv_v4["oof"]
print(f"v4 OOF: {roc_auc_score(y_train, lgbm_v4_oof):.4f}  (should be ~0.9452)")

Computing v4 baseline OOF for reference...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[422]	valid_0's auc: 0.948103	valid_0's binary_logloss: 0.266372
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[542]	valid_0's auc: 0.942917	valid_0's binary_logloss: 0.276489
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[349]	valid_0's auc: 0.939047	valid_0's binary_logloss: 0.287235
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[396]	valid_0's auc: 0.948063	valid_0's binary_logloss: 0.263957
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[420]	valid_0's auc: 0.948282	valid_0's binary_logloss: 0.266298
v4 OOF: 0.9452  (should be ~0.9452)


In [5]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd
import lightgbm as lgb
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2025
HERE = Path.cwd()
REPO = HERE.parent if HERE.name == "q5_kaggle" else HERE
DATA_DIR = REPO / "Assignment1" / "cs-610-assignment-1-question-5-2026"

train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
y_train = train["target"].values
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()

# best params from earlier Optuna run (v3 features)
best_params = {
    "learning_rate":    0.023244641257268547,
    "num_leaves":       90,
    "min_data_in_leaf": 14,
    "feature_fraction": 0.5042299812794246,
    "bagging_fraction": 0.9708522995554573,
    "bagging_freq":     4,
    "lambda_l1":        1.4345478190845018e-07,
    "lambda_l2":        3.437058206065732,
}

def featurize(df):
    out = pd.DataFrame(index=df.index)
    counts = ["favourites_count","followers_count","friends_count",
              "statuses_count","average_tweets_per_day","account_age_days"]
    for c in counts: out[c] = df[c].astype(float)
    for c in ["favourites_count","followers_count","friends_count","statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])
    for c in ["default_profile","default_profile_image","geo_enabled","verified"]:
        out[c] = df[c].astype(int)
    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"]    = (df["location"].notna() & (df["location"]!="unknown")).astype(int)
    out["has_url_bg"]      = df["profile_background_image_url"].notna().astype(int)
    out["desc_len"]  = df["description"].fillna("").str.len()
    out["sn_len"]    = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1))
    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")
    out["followers_per_friend"] = df["followers_count"]/df["friends_count"].clip(lower=1)
    out["statuses_per_day"]     = df["statuses_count"] /df["account_age_days"].clip(lower=1)
    ct = pd.to_datetime(df["created_at"])
    out["created_year"]  = ct.dt.year.astype(float)
    out["created_month"] = ct.dt.month.astype(float)
    out["created_dow"]   = ct.dt.dayofweek.astype(float)
    out["created_hour"]  = ct.dt.hour.astype(float)
    out["hour_sin"] = np.sin(2*np.pi*out["created_hour"]/24)
    out["hour_cos"] = np.cos(2*np.pi*out["created_hour"]/24)
    out["dow_sin"]  = np.sin(2*np.pi*out["created_dow"]/7)
    out["dow_cos"]  = np.cos(2*np.pi*out["created_dow"]/7)
    return out

X_v3_train = featurize(train); X_v3_test = featurize(test)
numeric_cols_v3 = [c for c in X_v3_train.columns if c != "lang"]

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=5, max_df=0.95,
                        lowercase=True, sublinear_tf=True, strip_accents="unicode")
Xt_train = tfidf.fit_transform(train["description"].fillna(""))
Xt_test  = tfidf.transform(test["description"].fillna(""))

oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_v3_train[["lang"]])
lang_test_oh  = oh.transform(X_v3_test[["lang"]])

num_train_v3 = sparse.csr_matrix(X_v3_train[numeric_cols_v3].fillna(0).values)
num_test_v3  = sparse.csr_matrix(X_v3_test[numeric_cols_v3].fillna(0).values)

X_combined_train_v3 = sparse.hstack([num_train_v3, lang_train_oh, Xt_train]).tocsr()
X_combined_test_v3  = sparse.hstack([num_test_v3,  lang_test_oh,  Xt_test ]).tocsr()

def lgbm_cv(X, y, n_splits=5, params=None, verbose=True):
    p = dict(n_estimators=2000, n_jobs=-1, verbose=-1)
    if params: p.update(params)
    p["random_state"] = p.get("random_state", RANDOM_STATE)
    take = lambda M, idx: M.iloc[idx] if hasattr(M, "iloc") else M[idx]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    oof = np.zeros(X.shape[0])
    for fold, (tr, va) in enumerate(skf.split(np.zeros(X.shape[0]), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr], eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        oof[va] = clf.predict_proba(take(X, va))[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        iters.append(clf.best_iteration_)
        if verbose: print(f"Fold {fold+1}: AUC={aucs[-1]:.4f}  iter={iters[-1]}")
    if verbose:
        print(f"\nMean fold AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"OOF AUC      : {roc_auc_score(y, oof):.4f}")
    return {"oof": oof, "aucs": aucs, "iters": iters}

In [6]:
import re

# Pattern → regex (case-insensitive applied separately). Mix of words, phrases, char patterns.
KEYWORDS = {
    "kw_bot":           r"\bbot\b",
    "kw_followback":    r"\b(follow\s*back|f4f|follow4follow|followback)\b",
    "kw_dm_me":         r"\bdm\s+me\b",
    "kw_click_here":    r"\bclick\s+(here|link|below|now)\b",
    "kw_visit_my":      r"\b(visit|go\s+to|check\s+out)\s+my\b",
    "kw_free":          r"\bfree\b",
    "kw_money":         r"\b(money|earn|income|cash)\b",
    "kw_crypto":        r"\b(crypto|bitcoin|btc|eth|trader|investor|trading|forex)\b",
    "kw_giveaway":      r"\b(win|winner|prize|gift|giveaway)\b",
    "kw_18plus":        r"\b(18\+|nsfw|only\s*fans?|xxx)\b",
    "kw_subscribe":     r"\b(subscribe|sub\s+to)\b",
    "kw_hack":          r"\bhack(ed|ing|er)?\b",
    "kw_url":           r"https?://",
    "kw_dollar_signs":  r"\$\$+",
    "kw_emoji_money":   r"[💰💸💵]",
    "kw_emoji_fire":    r"🔥",
    "kw_buy_now":       r"\b(buy\s+now|order\s+now|shop\s+now)\b",
    "kw_link_in_bio":   r"\blink\s+in\s+bio\b",
    "kw_cta_now":       r"\b(do\s+it\s+now|act\s+now|today\s+only|limited\s+time)\b",
    "kw_call":          r"\b(text|call|whatsapp)\s+(me|us)\b",
}

def keyword_features(text_series):
    s = text_series.fillna("").str.lower()
    out = pd.DataFrame(index=text_series.index)
    for name, pattern in KEYWORDS.items():
        out[name] = s.str.contains(pattern, regex=True, na=False).astype(int)
    out["kw_total"] = out.sum(axis=1)  # how many bot-y patterns hit
    return out

kw_train = keyword_features(train["description"])
kw_test  = keyword_features(test["description"])

# How discriminative is each keyword? (bot rate when present vs absent)
print("Bot rate by keyword presence (sorted by lift):")
diag = []
for c in kw_train.columns[:-1]:  # skip kw_total
    has = kw_train[c] == 1
    n_pos = has.sum()
    if n_pos < 20:
        continue
    rate_with = y_train[has].mean()
    rate_without = y_train[~has].mean()
    diag.append((c, n_pos, rate_with, rate_with - rate_without))
diag_df = pd.DataFrame(diag, columns=["keyword","n_present","bot_rate","lift"])
print(diag_df.sort_values("lift", ascending=False))

Bot rate by keyword presence (sorted by lift):
           keyword  n_present  bot_rate      lift
0           kw_bot        235  0.982979  0.653688
9          kw_hack         55  0.636364  0.301845
4         kw_money        173  0.601156  0.267772
5        kw_crypto        382  0.573298  0.241669
10          kw_url       2911  0.446925  0.125741
12   kw_emoji_fire         83  0.457831  0.123069
2      kw_visit_my         62  0.435484  0.100570
1    kw_followback         52  0.403846  0.068830
13         kw_call         51  0.235294 -0.100053
3          kw_free        158  0.234177 -0.101588
8     kw_subscribe         40  0.175000 -0.160397
7        kw_18plus         32  0.156250 -0.179121
6      kw_giveaway        110  0.118182 -0.217885
11  kw_emoji_money         33  0.090909 -0.244551


In [7]:
# Stack onto v3 dense features
X_v11_train_dense = pd.concat([X_v3_train, kw_train], axis=1)
X_v11_test_dense  = pd.concat([X_v3_test,  kw_test],  axis=1)
numeric_cols_v11 = [c for c in X_v11_train_dense.columns if c != "lang"]
num_train_v11 = sparse.csr_matrix(X_v11_train_dense[numeric_cols_v11].fillna(0).values)
num_test_v11  = sparse.csr_matrix(X_v11_test_dense[numeric_cols_v11].fillna(0).values)
X_combined_train_v11 = sparse.hstack([num_train_v11, lang_train_oh, Xt_train]).tocsr()
X_combined_test_v11  = sparse.hstack([num_test_v11,  lang_test_oh,  Xt_test ]).tocsr()
print(f"v11 shape: {X_combined_train_v11.shape}")

cv_v11 = lgbm_cv(X_combined_train_v11, y_train, params=best_params, verbose=False)
print(f"\nv4  OOF: {roc_auc_score(y_train, lgbm_v4_oof):.4f}")
print(f"v11 OOF (+ keywords): {roc_auc_score(y_train, cv_v11['oof']):.4f}")

v11 shape: (26206, 560)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[536]	valid_0's auc: 0.948448	valid_0's binary_logloss: 0.265159
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[395]	valid_0's auc: 0.942581	valid_0's binary_logloss: 0.277513
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[348]	valid_0's auc: 0.938425	valid_0's binary_logloss: 0.288916
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[368]	valid_0's auc: 0.948238	valid_0's binary_logloss: 0.264119
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[299]	valid_0's auc: 0.948349	valid_0's binary_logloss: 0.267517

v4  OOF: 0.9452
v11 OOF (+ keywords): 0.9451
